In [ ]:
import os
os.environ["GOOGLE_API_KEY"]='AIz ......'

In [ ]:
!pip install  youtube-transcript-api langchain-community \
                langchain-google-genai faiss-cpu tiktoken python-dotenv


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Step 1A- Indexing


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "Gfr50f6ZBvo"

try:
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id, languages=["en"])
    transcript = " ".join(chunk.text for chunk in fetched)
    print(transcript)




except  TranscriptsDisabled:
     print("No captions available for this video.")

# Text splitting


In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=3000,chunk_overlap=500)
chunks=splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[50]

# Embedding and store in vectorstore


In [ ]:
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
vector_store=FAISS.from_documents(embedding=embedding,documents=chunks)

In [ ]:
vector_store.index_to_docstore_id

# Retrival

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})#conduct similarity search and find top 5 similar

In [ ]:
retriever.invoke('What is deepmind')

# Augmentation

In [ ]:
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite',
    temperature=0.4)

In [ ]:
prompt=PromptTemplate(
    template="""
    You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}""",
      input_variables=['context','question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

# Generation


In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser=StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("can you summarize this video")